# Phase 3 — Lyrical Feature Extraction

**Input:** `outputs/merged_dataset.parquet` → `lyrics_text` column (2,075 multimodal tracks)  
**Output:** `outputs/lyrics_features.parquet`

## Three representations per track

| Representation | Method | Dims | Purpose |
|---|---|---|---|
| `lyrics_sbert` | `all-mpnet-base-v2` sentence embedding | 768 | Primary similarity signal |
| `lyrics_nrc` | NRC Emotion Lexicon (10 categories) | 10 | Interpretable emotion features |
| `lyrics_structural` | Lexical diversity + VADER + BERTopic | variable | Structural/linguistic signal |

## Performance notes
- SBERT: ~1–2 min for 2,075 tracks (batched encoding); checkpoint saved after first run
- NRC + VADER: ~20–30 s total (pure Python, no GPU needed)
- BERTopic: ~3–5 min one-time fit (reuses pre-computed SBERT embeddings)

## Required packages
```
pip install sentence-transformers nrclex nltk bertopic umap-learn hdbscan
```

In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nrclex import NRCLex
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR         = os.path.join(ROOT, 'outputs')
SBERT_CKPT_RAW  = os.path.join(OUT_DIR, 'lyrics_sbert_checkpoint_raw.npy')   # un-normalized (source of truth)
SBERT_CKPT      = os.path.join(OUT_DIR, 'lyrics_sbert_checkpoint.npy')        # kept for reference; derived from raw
BERTOPIC_PKL    = os.path.join(OUT_DIR, 'bertopic_model.pkl')
OUT_PATH        = os.path.join(OUT_DIR, 'lyrics_features.parquet')

NRC_CATEGORIES = [
    'anger', 'fear', 'anticipation', 'trust', 'surprise',
    'sadness', 'joy', 'disgust', 'positive', 'negative',
]

print('Libraries loaded')
print(f'Output path: {OUT_PATH}')

## 1. Load Dataset (Multimodal Tracks Only)

In [ ]:
df = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
lyrics_df = df[df['is_multimodal']].reset_index(drop=True)

print(f'Total tracks in merged dataset: {len(df)}')
print(f'Multimodal tracks (audio + lyrics): {len(lyrics_df)}')
print(f'\nQuadrant breakdown:')
print(lyrics_df['quadrant'].value_counts().sort_index().to_string())

# Validate lyrics text
missing = lyrics_df['lyrics_text'].isna().sum()
empty   = (lyrics_df['lyrics_text'].str.strip() == '').sum()
print(f'\nLyrics text: {missing} NaN, {empty} empty, {len(lyrics_df) - missing - empty} usable')

# Fill NaN with empty string to avoid downstream errors
lyrics_df = lyrics_df.copy()
lyrics_df['lyrics_text'] = lyrics_df['lyrics_text'].fillna('')

lyrics_texts = lyrics_df['lyrics_text'].tolist()
song_ids     = lyrics_df['song_id'].tolist()

print(f'\nSample lyrics preview (first track):')
print(lyrics_texts[0][:300], '...')

## 2. Representation 3a — SBERT Sentence Embeddings (768-dim)

Model: `all-mpnet-base-v2` from `sentence-transformers`  
Encodes the full lyric text as a single 768-dim dense vector via mean pooling.  
Checkpoint saved after first run — safe to re-run without re-encoding.

In [ ]:
if os.path.exists(SBERT_CKPT_RAW):
    sbert_raw_arr = np.load(SBERT_CKPT_RAW)
    print(f'Loaded raw SBERT checkpoint: shape={sbert_raw_arr.shape}')
else:
    import time
    print('Loading all-mpnet-base-v2 ...')
    t0 = time.time()
    sbert_full = SentenceTransformer('all-mpnet-base-v2')

    # all-mpnet-base-v2 pipeline: Transformer → Pooling → Normalize
    # Rebuild without the Normalize layer so we get mean-pooled vectors at their
    # natural magnitudes. Normalizing those vectors later recovers the standard SBERT
    # output exactly, giving us both raw and normalized from a single encode pass.
    raw_modules = [mod for _, mod in sbert_full._modules.items()
                   if type(mod).__name__ != 'Normalize']
    sbert_raw_model = SentenceTransformer(modules=raw_modules)
    print(f'Model ready in {time.time()-t0:.1f}s  '
          f'| pipeline: {[type(m).__name__ for m in raw_modules]}')

    print('Encoding lyrics (raw, un-normalized) ...')
    sbert_raw_arr = sbert_raw_model.encode(
        lyrics_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype(np.float32)

    np.save(SBERT_CKPT_RAW, sbert_raw_arr)
    print(f'Raw checkpoint saved: {SBERT_CKPT_RAW}')

# Derive normalized version from raw — no second encode pass needed
raw_norms = np.linalg.norm(sbert_raw_arr, axis=1, keepdims=True)
sbert_arr = (sbert_raw_arr / raw_norms).astype(np.float32)

assert sbert_raw_arr.shape == (len(lyrics_df), 768), f'Unexpected shape: {sbert_raw_arr.shape}'
assert sbert_arr.shape == sbert_raw_arr.shape

raw_norm_vals  = np.linalg.norm(sbert_raw_arr, axis=1)
norm_norm_vals = np.linalg.norm(sbert_arr, axis=1)
nan_rows       = np.isnan(sbert_raw_arr).any(axis=1).sum()

print(f'\nSBERT raw:         shape={sbert_raw_arr.shape}  dtype={sbert_raw_arr.dtype}')
print(f'  NaN rows:        {nan_rows}')
print(f'  norms:           mean={raw_norm_vals.mean():.4f}  std={raw_norm_vals.std():.4f}'
      f'  min={raw_norm_vals.min():.4f}  max={raw_norm_vals.max():.4f}')
print(f'  NOT unit-norm:   {not np.allclose(raw_norm_vals, 1.0, atol=1e-2)}')
print(f'\nSBERT normalized:  shape={sbert_arr.shape}')
print(f'  norms:           mean={norm_norm_vals.mean():.6f}  std={norm_norm_vals.std():.6f}  (should be 1.0)')

## 3. Representation 3b — NRC Emotion Lexicon (10-dim)

Uses the NRC Word-Emotion Association Lexicon via the `nrclex` package.  
Raw per-emotion word counts are normalized by total affect-word count → 10-dim float vector.  
Categories: `anger, fear, anticipation, trust, surprise, sadness, joy, disgust, positive, negative`

API compatibility: uses `raw_emotion_scores` (NRCLex ≥ 3.x) or `top_emotions` (NRCLex 1.x/2.x).

In [ ]:
def extract_nrc(text: str) -> np.ndarray:
    """10-dim NRC emotion vector, normalized by total affect-word count.

    NRCLex 4.x API: constructor takes no text; text is loaded via load_token_list().
    Using simple whitespace split (no TextBlob/punkt dependency).
    """
    if not text.strip():
        return np.zeros(10, dtype=np.float32)
    nrc = NRCLex()                            # v4.x: no text in constructor
    nrc.load_token_list(text.lower().split()) # bypass TextBlob/punkt entirely
    freq  = nrc.affect_frequencies            # {emotion: count}
    total = max(sum(freq.values()), 1)
    return np.array([freq.get(cat, 0) / total for cat in NRC_CATEGORIES], dtype=np.float32)

# Smoke-test: should show non-zero values for a clearly emotional sentence
_test_text = 'I love the way you make me feel so happy and joyful today'
_test = extract_nrc(_test_text)
print(f'Smoke-test: shape={_test.shape}, sum={_test.sum():.3f}, any NaN={np.isnan(_test).any()}')
print(f'  {dict(zip(NRC_CATEGORIES, _test.round(3)))}')
assert _test.sum() > 0, 'NRC returned all zeros — API mismatch, check nrclex version'
print()

print('Extracting NRC Emotion Lexicon features ...')
nrc_list = [
    extract_nrc(text)
    for text in tqdm(lyrics_texts, desc='NRC', ncols=80)
]
nrc_arr = np.vstack(nrc_list)  # (N, 10)

zero_rows = (nrc_arr.sum(axis=1) == 0).sum()
print(f'\nNRC array: shape={nrc_arr.shape}, dtype={nrc_arr.dtype}')
print(f'NaN rows:       {np.isnan(nrc_arr).any(axis=1).sum()}')
print(f'All-zero rows:  {zero_rows}  (tracks with no NRC-matched words)')
print(f'\nCorpus-level mean per category:')
for cat, val in zip(NRC_CATEGORIES, nrc_arr.mean(axis=0)):
    bar = '|' * int(val * 200)
    print(f'  {cat:15s}: {val:.4f}  {bar}')

## 4. Representation 3c — Structural / Linguistic Features

Three scalar signals computed from the raw text:

| Feature | Description |
|---|---|
| `lexical_diversity` | Type-token ratio (unique words / total words) |
| `vader_compound` | Mean VADER compound score over lines (−1 = most negative, +1 = most positive) |
| `topic_id` + `topic_probs` | BERTopic topic assignment + soft probability vector (next cell) |

In [ ]:
sid = SentimentIntensityAnalyzer()

def lexical_diversity(text: str) -> float:
    tokens = text.lower().split()
    return float(len(set(tokens)) / len(tokens)) if tokens else 0.0

def mean_vader_compound(text: str) -> float:
    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    if not lines:
        return 0.0
    scores = [sid.polarity_scores(ln)['compound'] for ln in lines]
    return float(np.mean(scores))

print('Extracting VADER + lexical diversity ...')
lex_divs     = []
vader_scores = []
for text in tqdm(lyrics_texts, desc='Structural', ncols=80):
    lex_divs.append(lexical_diversity(text))
    vader_scores.append(mean_vader_compound(text))

lex_divs     = np.array(lex_divs,     dtype=np.float32)
vader_scores = np.array(vader_scores, dtype=np.float32)

quad_labels = lyrics_df['quadrant'].values

print(f'\nLexical diversity: mean={lex_divs.mean():.3f}, std={lex_divs.std():.3f}')
print(f'  range: [{lex_divs.min():.3f}, {lex_divs.max():.3f}]')

print(f'\nVADER compound score by quadrant:')
q_labels_map = {'Q1': 'Happy', 'Q2': 'Angry', 'Q3': 'Sad', 'Q4': 'Calm'}
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    mask = quad_labels == q
    v = vader_scores[mask]
    print(f'  {q} ({q_labels_map[q]:5s}): mean={v.mean():.3f}, std={v.std():.3f}')

## 5. BERTopic — Unsupervised Topic Distribution

Fits BERTopic on the full lyrics corpus using the **pre-computed SBERT embeddings** (avoids double encoding).  
UMAP reduces 768-dim → 5-dim; HDBSCAN clusters into topics.  
With `calculate_probabilities=True`, each document gets a soft membership vector over all topics.

**No mood/genre labels are used here — fully unsupervised.**

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

if os.path.exists(BERTOPIC_PKL):
    print('Loading cached BERTopic model ...')
    with open(BERTOPIC_PKL, 'rb') as fh:
        topic_model = pickle.load(fh)
    topics, probs = topic_model.transform(lyrics_texts, embeddings=sbert_arr)
    topics = np.array(topics, dtype=np.int32)
    print('BERTopic model loaded and transformed.')
else:
    umap_model = UMAP(
        n_components=5,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        random_state=42,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=10,   # lower → more topics; was 20 (produced only 3)
        min_samples=5,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=True,
        verbose=True,
    )
    print('Fitting BERTopic (reusing SBERT embeddings) ...')
    topics, probs = topic_model.fit_transform(lyrics_texts, embeddings=sbert_arr)
    topics = np.array(topics, dtype=np.int32)

    with open(BERTOPIC_PKL, 'wb') as fh:
        pickle.dump(topic_model, fh)
    print(f'BERTopic model saved: {BERTOPIC_PKL}')

# Summarise results
topic_info = topic_model.get_topic_info()
n_topics   = len(topic_info[topic_info['Topic'] != -1])
n_outliers = int((topics == -1).sum())

print(f'\nBERTopic: {n_topics} topics found')
print(f'Outlier documents (topic -1): {n_outliers} / {len(topics)}')
print(f'\nTop 10 topics by size:')
cols = [c for c in ['Topic', 'Count', 'Name', 'Representation'] if c in topic_info.columns]
print(topic_info[cols].head(11).to_string(index=False))

In [ ]:
# Build per-document topic probability vectors
if probs is not None and np.ndim(probs) == 2:
    # Soft HDBSCAN membership probabilities: shape (n_docs, n_topics)
    # Row-normalize so each document's vector sums to 1
    row_sums = probs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    probs_norm = (probs / row_sums).astype(np.float32)
    topic_probs_list = probs_norm.tolist()
    print(f'Soft topic probs: shape={probs.shape}, normalized')
else:
    # Fallback: one-hot over hard topic assignment (outliers → all-zeros)
    print('Falling back to one-hot topic encoding')
    unique_topics  = sorted([t for t in set(topics.tolist()) if t != -1])
    topic_to_idx   = {t: i for i, t in enumerate(unique_topics)}
    n_t            = len(unique_topics)
    probs_onehot   = np.zeros((len(topics), n_t), dtype=np.float32)
    for i, t in enumerate(topics):
        if t != -1:
            probs_onehot[i, topic_to_idx[t]] = 1.0
    topic_probs_list = probs_onehot.tolist()
    print(f'One-hot topic probs: shape={probs_onehot.shape}')

print(f'topic_probs_list[0] length: {len(topic_probs_list[0])}')

## 6. Assemble and Save `lyrics_features.parquet`

The `lyrics_structural` column is stored as a JSON string to keep the parquet schema flat.  
Load it back with `.apply(json.loads)` in later phases.

In [ ]:
structural_dicts = [
    {
        'lexical_diversity': float(ld),
        'vader_compound':    float(vc),
        'topic_id':          int(tid),
        'topic_probs':       tp,
    }
    for ld, vc, tid, tp in zip(lex_divs, vader_scores, topics, topic_probs_list)
]

feat_df = pd.DataFrame({
    'song_id':           song_ids,
    'lyrics_sbert':      [sbert_arr[i].tolist()     for i in range(len(sbert_arr))],
    'lyrics_sbert_raw':  [sbert_raw_arr[i].tolist() for i in range(len(sbert_raw_arr))],
    'lyrics_nrc':        [nrc_arr[i].tolist()        for i in range(len(nrc_arr))],
    'lyrics_structural': [json.dumps(d)              for d in structural_dicts],
})

# Merge quadrant + AV labels for convenience in later phases
meta_cols = df[['song_id', 'quadrant', 'arousal', 'valence']]
feat_df   = feat_df.merge(meta_cols, on='song_id', how='left')

feat_df.to_parquet(OUT_PATH, index=False, engine='pyarrow')
size_mb = os.path.getsize(OUT_PATH) / 1e6

print(f'Saved: {OUT_PATH}  ({size_mb:.1f} MB)')
print(f'Rows: {len(feat_df)}')
print(f'Columns: {feat_df.columns.tolist()}')

## 7. Validation

In [ ]:
feat = pd.read_parquet(OUT_PATH)
print(f'Reloaded: {feat.shape}')
print(f'Columns: {feat.columns.tolist()}')

sbert_loaded     = np.vstack(feat['lyrics_sbert'].values).astype(np.float32)
sbert_raw_loaded = np.vstack(feat['lyrics_sbert_raw'].values).astype(np.float32)
nrc_loaded       = np.vstack(feat['lyrics_nrc'].values).astype(np.float32)

print(f'\nSBERT normalized: shape={sbert_loaded.shape},     NaN rows={np.isnan(sbert_loaded).any(axis=1).sum()}')
print(f'SBERT raw:        shape={sbert_raw_loaded.shape},  NaN rows={np.isnan(sbert_raw_loaded).any(axis=1).sum()}')
print(f'NRC:              shape={nrc_loaded.shape},          NaN rows={np.isnan(nrc_loaded).any(axis=1).sum()}')

assert sbert_loaded.shape[1]     == 768, 'SBERT normalized dim mismatch'
assert sbert_raw_loaded.shape[1] == 768, 'SBERT raw dim mismatch'
assert nrc_loaded.shape[1]       == 10,  'NRC dim mismatch'

# Norm checks
sbert_norms     = np.linalg.norm(sbert_loaded,     axis=1)
sbert_raw_norms = np.linalg.norm(sbert_raw_loaded, axis=1)
print(f'\nSBERT norms:     mean={sbert_norms.mean():.6f}  std={sbert_norms.std():.6f}  (should be 1.0)')
print(f'SBERT raw norms: mean={sbert_raw_norms.mean():.4f}  std={sbert_raw_norms.std():.4f}'
      f'  min={sbert_raw_norms.min():.4f}  max={sbert_raw_norms.max():.4f}')
print(f'  lyrics_sbert is unit-normalized:         {np.allclose(sbert_norms, 1.0, atol=1e-5)}')
print(f'  lyrics_sbert_raw is NOT unit-normalized: {not np.allclose(sbert_raw_norms, 1.0, atol=1e-2)}')

# Round-trip: normalize(sbert_raw) should recover sbert exactly
recovered = sbert_raw_loaded / sbert_raw_norms[:, None]
match = np.allclose(recovered, sbert_loaded, atol=1e-5)
print(f'  normalize(sbert_raw) == sbert across all {len(feat)} tracks: {match}')
print(f'  Max abs error: {np.abs(recovered - sbert_loaded).max():.2e}')

# Sample structural parse
sample = json.loads(feat['lyrics_structural'].iloc[0])
print(f'\nSample structural (row 0):')
print(f'  lexical_diversity : {sample["lexical_diversity"]:.4f}')
print(f'  vader_compound    : {sample["vader_compound"]:.4f}')
print(f'  topic_id          : {sample["topic_id"]}')
print(f'  topic_probs dims  : {len(sample["topic_probs"])}')

print(f'\nCoverage by quadrant:')
print(feat.groupby('quadrant')['song_id'].count().sort_index().to_string())

print('\nAll validation checks passed.')

## 8. EDA — Do Lyrical Features Separate Emotion Quadrants?

Three sanity-check plots:
1. PCA of SBERT + NRC features colored by quadrant
2. NRC emotion heatmap (mean per quadrant)
3. VADER compound score distribution per quadrant

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

QUADRANT_COLORS = {'Q1': '#f4a261', 'Q2': '#e63946', 'Q3': '#457b9d', 'Q4': '#2a9d8f'}
QUADRANT_LABELS = {'Q1': 'Happy', 'Q2': 'Angry', 'Q3': 'Sad', 'Q4': 'Calm'}

qlabels = feat['quadrant'].values

def pca_scatter(ax, X, labels, title):
    X_s = StandardScaler().fit_transform(X)
    pcs = PCA(n_components=2, random_state=42).fit_transform(X_s)
    for q, color in QUADRANT_COLORS.items():
        mask = labels == q
        ax.scatter(pcs[mask, 0], pcs[mask, 1], c=color, s=8, alpha=0.45,
                   label=f'{q} {QUADRANT_LABELS[q]}')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.legend(fontsize=8, markerscale=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('PCA of Lyrical Features — Quadrant Separation Sanity Check',
             fontsize=13, fontweight='bold')

pca_scatter(axes[0], sbert_loaded, qlabels, 'SBERT embeddings (768-dim → PC1/PC2)')
pca_scatter(axes[1], nrc_loaded,   qlabels, 'NRC emotion features (10-dim → PC1/PC2)')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lyrics_pca.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lyrics_pca.png')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

nrc_by_q = np.array([
    nrc_loaded[qlabels == q].mean(axis=0)
    for q in ['Q1', 'Q2', 'Q3', 'Q4']
])

im = ax.imshow(nrc_by_q, aspect='auto', cmap='YlOrRd', vmin=0)
ax.set_xticks(range(10))
ax.set_xticklabels(NRC_CATEGORIES, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(4))
ax.set_yticklabels([f'Q{i+1} ({QUADRANT_LABELS[f"Q{i+1}"]})' for i in range(4)], fontsize=10)

for i in range(4):
    for j in range(10):
        ax.text(j, i, f'{nrc_by_q[i, j]:.3f}', ha='center', va='center', fontsize=8)

ax.set_title('Mean NRC Affect Frequencies by Emotion Quadrant', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Affect frequency')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lyrics_nrc_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lyrics_nrc_heatmap.png')

In [ ]:
struct_col   = feat['lyrics_structural'].apply(json.loads)
vader_all    = np.array([d['vader_compound']    for d in struct_col], dtype=np.float32)
lex_div_all  = np.array([d['lexical_diversity'] for d in struct_col], dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Structural Lyrical Features by Emotion Quadrant', fontsize=13, fontweight='bold')

# VADER distribution
ax = axes[0]
for q, color in QUADRANT_COLORS.items():
    mask = qlabels == q
    ax.hist(vader_all[mask], bins=30, alpha=0.55, color=color,
            label=f'{q} {QUADRANT_LABELS[q]}', density=True)
ax.axvline(0, color='black', linestyle='--', linewidth=0.8, label='Neutral')
ax.set_xlabel('Mean VADER Compound Score', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('VADER Sentiment Distribution', fontsize=11)
ax.legend(fontsize=9)

# Lexical diversity distribution
ax = axes[1]
for q, color in QUADRANT_COLORS.items():
    mask = qlabels == q
    ax.hist(lex_div_all[mask], bins=30, alpha=0.55, color=color,
            label=f'{q} {QUADRANT_LABELS[q]}', density=True)
ax.set_xlabel('Lexical Diversity (Type-Token Ratio)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Lexical Diversity Distribution', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lyrics_structural_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lyrics_structural_dist.png')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

print('Quadrant classification accuracy (5-fold CV) — sanity check only:')
print('  Chance level (4 classes): 0.250')
print()

X_sbert = StandardScaler().fit_transform(sbert_loaded)
X_nrc   = StandardScaler().fit_transform(nrc_loaded)
y       = feat['quadrant'].values

clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

scores_sbert = cross_val_score(clf, X_sbert, y, cv=5, scoring='accuracy')
print(f'  SBERT (768-dim): {scores_sbert.mean():.3f} ± {scores_sbert.std():.3f}')

scores_nrc = cross_val_score(clf, X_nrc, y, cv=5, scoring='accuracy')
print(f'  NRC   (10-dim):  {scores_nrc.mean():.3f} ± {scores_nrc.std():.3f}')

X_fused = np.hstack([X_sbert, X_nrc])
scores_fused = cross_val_score(clf, X_fused, y, cv=5, scoring='accuracy')
print(f'  SBERT+NRC fused: {scores_fused.mean():.3f} ± {scores_fused.std():.3f}')

## Phase 3 Complete

**Output:** `outputs/lyrics_features.parquet`

| Column | Type | Dims | Normalized | Description |
|---|---|---|---|---|
| `song_id` | str | — | — | Track identifier |
| `quadrant` | str | — | — | Q1–Q4 emotion label |
| `arousal` | float | — | — | Continuous arousal score |
| `valence` | float | — | — | Continuous valence score |
| `lyrics_sbert` | list[float] | 768 | Yes (norm=1.0) | SBERT embedding — lyrics-only FAISS index; late fusion |
| `lyrics_sbert_raw` | list[float] | 768 | No | SBERT mean-pool before Normalize — early fusion concatenation with `audio_clap_raw` |
| `lyrics_nrc` | list[float] | 10 | No | NRC affect frequencies |
| `lyrics_structural` | JSON str | variable | — | `{lexical_diversity, vader_compound, topic_id, topic_probs}` |

**Why two SBERT columns?**  
`all-mpnet-base-v2` has a Normalize module at the end of its pipeline, so `encode()` always
returns L2-normalized vectors. Concatenating unit-normalized CLAP and SBERT vectors makes
early fusion mathematically identical to late fusion at α=0.5. `lyrics_sbert_raw` is the
mean-pooled output *before* the Normalize layer, extracted by rebuilding the model pipeline
without that final step and deriving the normalized version via `raw / ‖raw‖`.

**Loading in later phases:**
```python
import json, numpy as np
feat = pd.read_parquet('outputs/lyrics_features.parquet')
sbert_arr     = np.vstack(feat['lyrics_sbert'].values)        # (2075, 768), unit-normalized
sbert_raw_arr = np.vstack(feat['lyrics_sbert_raw'].values)    # (2075, 768), un-normalized
nrc_arr       = np.vstack(feat['lyrics_nrc'].values)          # (2075, 10)
struct        = feat['lyrics_structural'].apply(json.loads)
topic_probs   = np.vstack([d['topic_probs'] for d in struct]) # (2075, n_topics)
vader_arr     = np.array([d['vader_compound'] for d in struct])
```

**Checkpoints:**
- `outputs/lyrics_sbert_checkpoint_raw.npy` — raw (un-normalized) SBERT vectors; source of truth
- `outputs/bertopic_model.pkl` — fitted BERTopic model

**Next:** Phase 4 — Similarity Engine (`phase4_similarity_engine.ipynb`)